# 딥러닝 입문: 퍼셉트론부터 TensorFlow와 PyTorch까지

이 노트북에서는 하나의 작은 이진 분류 문제를 사용해 퍼셉트론, 다층 퍼셉트론(MLP), 활성화 함수, TensorFlow Keras, PyTorch의 기본 흐름을 단계별로 학습합니다.

## 1. 퍼셉트론의 핵심

퍼셉트론은 입력값에 가중치를 곱하고 bias를 더한 뒤 활성화 함수에 통과시키는 가장 단순한 인공 뉴런입니다.

$$z = w_1x_1 + w_2x_2 + b$$

초기 퍼셉트론은 계단 함수를 사용해 0 또는 1을 출력했습니다. 선형적으로 나눌 수 있는 AND, OR 문제에는 잘 동작하지만 XOR처럼 직선 하나로 나눌 수 없는 문제는 해결하지 못합니다.

In [ ]:
import numpy as np

def step_function(value):
    return (value >= 0).astype(int)

def perceptron_forward(inputs, weights, bias):
    weighted_sum = inputs @ weights + bias
    return step_function(weighted_sum)

# AND 게이트를 구현하는 퍼셉트론
and_inputs = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
and_weights = np.array([1.0, 1.0])
and_bias = -1.5

print(perceptron_forward(and_inputs, and_weights, and_bias))

## 2. 활성화 함수

활성화 함수는 신경망에 비선형성을 추가합니다. 활성화 함수가 없으면 여러 층을 쌓아도 결국 하나의 선형 변환과 같아 복잡한 패턴을 학습하기 어렵습니다.

- **Sigmoid**: 출력 범위가 0~1이라 이진 분류의 확률 출력에 자주 사용합니다.
- **tanh**: 출력 범위가 -1~1입니다.
- **ReLU**: 음수는 0, 양수는 그대로 통과시키며 은닉층에서 널리 사용됩니다.
- **Softmax**: 여러 클래스의 점수를 확률처럼 변환하고 합이 1이 되게 합니다.

In [ ]:
import matplotlib.pyplot as plt

values = np.linspace(-5, 5, 200)
sigmoid = 1 / (1 + np.exp(-values))
tanh = np.tanh(values)
relu = np.maximum(0, values)

plt.figure(figsize=(9, 5))
plt.plot(values, sigmoid, label='sigmoid')
plt.plot(values, tanh, label='tanh')
plt.plot(values, relu, label='ReLU')
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.title('Activation Functions')
plt.legend()
plt.show()

## 3. 다층 퍼셉트론(MLP)

MLP는 입력층, 하나 이상의 은닉층, 출력층으로 구성됩니다. 은닉층의 가중치를 학습하고, 오차를 뒤에서 앞으로 전달하는 **역전파(backpropagation)**와 경사하강법으로 가중치를 업데이트합니다.

은닉층에서는 보통 ReLU를 사용하고, 이진 분류의 출력층에서는 sigmoid를 사용합니다. 아래 코드는 학습 없이 순전파의 모양만 확인하는 예제입니다.

In [ ]:
np.random.seed(42)
inputs = np.array([[0.2, 0.8], [0.9, 0.1]])
hidden_weights = np.random.randn(2, 3)
hidden_bias = np.zeros(3)
output_weights = np.random.randn(3, 1)
output_bias = np.zeros(1)

hidden_values = np.maximum(0, inputs @ hidden_weights + hidden_bias)
output_logits = hidden_values @ output_weights + output_bias
output_probability = 1 / (1 + np.exp(-output_logits))

print('은닉층 출력 모양:', hidden_values.shape)
print('출력 확률:', output_probability.ravel())

## 4. 실습 데이터 준비

두 개의 입력 특성으로 두 클래스를 구분하는 간단한 데이터를 만듭니다. 학습 중에는 train 데이터만 사용하고, 마지막에 test 데이터로 일반화 성능을 확인합니다.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

X, y = make_moons(n_samples=400, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='coolwarm')
plt.title('Training Data')
plt.show()
print('train:', X_train.shape, 'test:', X_test.shape)

## 5. TensorFlow Keras로 MLP 만들기

Keras에서는 `Sequential`에 층을 순서대로 쌓고 `compile`로 손실 함수와 optimizer를 정한 뒤 `fit`으로 학습합니다. `binary_crossentropy`는 이진 분류에 적합한 손실 함수입니다.

In [ ]:
try:
    import tensorflow as tf

    tf.random.set_seed(42)
    tensorflow_model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(2,)),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(8, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    tensorflow_model.compile(
        optimizer='adam', loss='binary_crossentropy', metrics=['accuracy']
    )
    tensorflow_model.fit(X_train, y_train, epochs=30, batch_size=32, verbose=0)
    loss, accuracy = tensorflow_model.evaluate(X_test, y_test, verbose=0)
    print(f'TensorFlow test accuracy: {accuracy:.3f}')
    tensorflow_prediction = (tensorflow_model.predict(X_test, verbose=0) >= 0.5).astype(int).ravel()
    print(confusion_matrix(y_test, tensorflow_prediction))
except ImportError:
    print('TensorFlow가 설치되어 있지 않습니다. pip install tensorflow 로 설치하세요.')

## 6. PyTorch로 같은 MLP 만들기

PyTorch에서는 `nn.Module`로 모델 구조를 직접 정의하고, `DataLoader`로 batch를 공급합니다. 학습 루프에서 예측, loss 계산, `backward`, optimizer 업데이트를 명시적으로 작성한다는 점이 Keras와 다른 핵심 특징입니다.

In [ ]:
try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset

    torch.manual_seed(42)
    train_dataset = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
    )
    test_tensor = torch.tensor(X_test, dtype=torch.float32)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    class MLP(nn.Module):
        def __init__(self):
            super().__init__()
            self.layers = nn.Sequential(
                nn.Linear(2, 16), nn.ReLU(),
                nn.Linear(16, 8), nn.ReLU(),
                nn.Linear(8, 1)
            )

        def forward(self, features):
            return self.layers(features)

    pytorch_model = MLP()
    loss_function = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(pytorch_model.parameters(), lr=0.01)

    for epoch in range(30):
        for features, targets in train_loader:
            logits = pytorch_model(features)
            loss = loss_function(logits, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    with torch.no_grad():
        test_logits = pytorch_model(test_tensor)
        pytorch_prediction = (torch.sigmoid(test_logits) >= 0.5).int().numpy().ravel()
    print(f'PyTorch test accuracy: {accuracy_score(y_test, pytorch_prediction):.3f}')
    print(confusion_matrix(y_test, pytorch_prediction))
except ImportError:
    print('PyTorch가 설치되어 있지 않습니다. pip install torch 로 설치하세요.')

## 7. TensorFlow와 PyTorch 비교

| 항목 | TensorFlow Keras | PyTorch |
| --- | --- | --- |
| 모델 작성 | `Sequential`, `Model`로 간결하게 구성 | `nn.Module`에서 구조를 직접 정의 |
| 학습 방식 | `compile`과 `fit`이 학습 과정을 추상화 | forward, loss, backward, update를 직접 작성 |
| 장점 | 빠른 프로토타이핑과 배포 생태계 | 유연한 디버깅과 연구용 실험 |
| 먼저 익힐 것 | layer, loss, optimizer, callback | tensor, module, DataLoader, training loop |

학습 순서는 `퍼셉트론 -> 활성화 함수 -> MLP 순전파 -> loss와 optimizer -> validation/test 평가`가 좋습니다. 프레임워크 문법보다 각 층의 입력·출력 shape와 데이터가 어떻게 흐르는지 먼저 확인하세요.